# MLOps-Lite — drive the whole platform from a notebook

Everything the operator console does is just a call to the **gateway API** (`http://localhost:8080`,
authenticated with `X-API-Key`). This notebook does **inference, streaming, vision, dataset
versioning, LoRA fine-tuning, promotion, and drift checks** — all server-side. The notebook itself
only needs `requests` (the GPU/model work happens in the platform's daemons).

**Prereqs**
- Platform up: `./scripts/up_all.ps1` (gateway + native daemons healthy).
- API key available: either `GATEWAY_API_KEY` in the environment, or `GATEWAY_API_KEYS` in the
  repo `.env` (this notebook reads `../.env` automatically).
- `pip install requests`

**One-model-in-VRAM (Principle II):** training refuses while a model is resident in serving, and
vice-versa — you'll get a clear `409`. The fine-tune cell below waits for the GPU to free itself.

In [ ]:
import os, base64, json, time, requests

# Base URL. localhost works from Windows AND WSL (the daemons use the same forwarding). If a WSL
# notebook can't reach it, set MLOPS_GATEWAY to the Windows host IP.
BASE_URL = os.environ.get("MLOPS_GATEWAY", "http://localhost:8080")

def _load_key():
    k = os.environ.get("GATEWAY_API_KEY")
    if k:
        return k
    here = os.getcwd()
    for d in (here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))):
        p = os.path.join(d, ".env")
        if os.path.isfile(p):
            for line in open(p, encoding="utf-8"):
                if line.startswith("GATEWAY_API_KEYS="):
                    return line.split("=", 1)[1].strip().split(",")[0]
    raise RuntimeError("No API key. Set GATEWAY_API_KEY, or run from inside the repo so ../.env is found.")

API_KEY = _load_key()


class MLOps:
    """Thin client over the MLOps-Lite gateway — the same endpoints the operator console uses."""

    def __init__(self, base_url=BASE_URL, api_key=API_KEY):
        self.base = base_url.rstrip("/")
        self.h = {"X-API-Key": api_key, "Content-Type": "application/json"}

    # ---------- health ----------
    def health(self):
        return requests.get(f"{self.base}/platform/health", timeout=5).json()

    def serving_resident(self):
        # read one /platform/events frame to see if a model is currently in VRAM
        with requests.get(f"{self.base}/platform/events", headers=self.h, stream=True, timeout=10) as r:
            for line in r.iter_lines(decode_unicode=True):
                if line and line.startswith("data:"):
                    ev = json.loads(line[5:].strip())
                    if ev.get("event") == "state":
                        s = ev.get("serving") or {}
                        return bool(s.get("resident"))
        return None

    # ---------- inference ----------
    def infer(self, prompt, max_tokens=256, temperature=0.7):
        r = requests.post(f"{self.base}/infer", headers=self.h,
                          json={"prompt": prompt, "max_tokens": max_tokens, "temperature": temperature},
                          timeout=300)
        r.raise_for_status()
        return r.json()

    def infer_stream(self, prompt, max_tokens=256, temperature=0.7):
        with requests.post(f"{self.base}/infer/stream", headers=self.h,
                           json={"prompt": prompt, "max_tokens": max_tokens, "temperature": temperature},
                           stream=True, timeout=300) as r:
            r.raise_for_status()
            for line in r.iter_lines(decode_unicode=True):
                if line and line.startswith("data:"):
                    yield json.loads(line[5:].strip())

    # ---------- vision ----------
    def classify_bytes(self, data):
        b64 = base64.b64encode(data).decode()
        r = requests.post(f"{self.base}/vision/classify", headers=self.h, json={"image_b64": b64}, timeout=60)
        r.raise_for_status()
        return r.json()

    def classify(self, image_path):
        return self.classify_bytes(open(image_path, "rb").read())

    # ---------- registry ----------
    def models(self):
        return requests.get(f"{self.base}/models", headers=self.h, timeout=15).json()["models"]

    def model(self, name):
        return requests.get(f"{self.base}/models/{name}", headers=self.h, timeout=15).json()

    def promote(self, name, version):
        r = requests.post(f"{self.base}/models/{name}/promote", headers=self.h,
                          json={"version": str(version)}, timeout=15)
        r.raise_for_status()
        return r.json()

    # ---------- datasets ----------
    def datasets(self):
        return requests.get(f"{self.base}/datasets", headers=self.h, timeout=15).json()["datasets"]

    def register_dataset(self, name, content, fmt=None):
        if isinstance(content, str):
            content = content.encode()
        r = requests.post(f"{self.base}/datasets", headers=self.h,
                          json={"name": name, "content_b64": base64.b64encode(content).decode(), "format": fmt},
                          timeout=30)
        r.raise_for_status()
        return r.json()

    # ---------- training / fine-tune ----------
    def launch_run(self, dataset_name, dataset_version, output_name,
                   steps=10, lora_r=8, seed=0, base_model=None, wait_for_gpu=True):
        body = {"dataset_name": dataset_name, "dataset_version": dataset_version,
                "output_name": output_name, "steps": steps, "lora_r": lora_r, "seed": seed}
        if base_model:
            body["base_model"] = base_model
        for attempt in range(20):
            r = requests.post(f"{self.base}/runs", headers=self.h, json=body, timeout=15)
            if r.status_code == 409 and wait_for_gpu:
                print(f"  GPU busy (one model in VRAM) — waiting for release... [{attempt}]")
                time.sleep(15)
                continue
            r.raise_for_status()
            return r.json()
        raise RuntimeError("GPU stayed busy; a model is resident in serving (idle-release ~120s).")

    def run(self, run_id):
        return requests.get(f"{self.base}/runs/{run_id}", headers=self.h, timeout=15).json()

    def stream_run(self, run_id):
        with requests.get(f"{self.base}/runs/{run_id}/events", headers=self.h, stream=True, timeout=3600) as r:
            r.raise_for_status()
            for line in r.iter_lines(decode_unicode=True):
                if line and line.startswith("data:"):
                    yield json.loads(line[5:].strip())

    # ---------- monitoring ----------
    def drift_check(self, ref_name, ref_version, cur_name, cur_version, threshold=0.25, retrain=None):
        body = {"reference": {"name": ref_name, "version": ref_version},
                "current": {"name": cur_name, "version": cur_version}, "threshold": threshold}
        if retrain:
            body["retrain"] = retrain
        r = requests.post(f"{self.base}/monitor/check", headers=self.h, json=body, timeout=120)
        r.raise_for_status()
        return r.json()


mlops = MLOps()
print("gateway:", BASE_URL, "| key:", API_KEY[:6] + "...")

In [ ]:
# health: all daemons reachable through the gateway?
mlops.health()

## 1 · Inference (on-demand — the first call cold-loads the model onto the GPU)

In [ ]:
r = mlops.infer("In one sentence, what is MLOps?", max_tokens=64, temperature=0.2)
print(r["text"].strip())
print(f"\nmodel={r['model']}  load_ms={r['load_ms']}  infer_ms={r['infer_ms']}  registry_version={r['registry_version']}")

### Streaming (token-by-token, via SSE)

In [ ]:
for ev in mlops.infer_stream("List three benefits of a local MLOps platform.", max_tokens=120):
    if ev.get("event") == "start":
        print(f"[model loaded in {ev['load_ms']} ms]\n")
    elif ev.get("event") == "token":
        print(ev["text"], end="", flush=True)
    elif ev.get("event") == "done":
        print(f"\n\n[{ev['tokens']} tokens in {ev['infer_ms']} ms]")

## 2 · Vision (top-5 image classification, CPU)

In [ ]:
# a tiny 1x1 PNG so this runs without a file; swap for: mlops.classify("path/to/image.png")
TINY_PNG = base64.b64decode(
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAAC0lEQVR42mP8z8BQDwAEhQGAhKmMIQAAAABJRU5ErkJggg==")
res = mlops.classify_bytes(TINY_PNG)
for p in res["predictions"]:
    print(f"  {p['score']*100:5.1f}%  {p['label']}")
print("device:", res["device"], "| model:", res["model"])

## 3 · Datasets (immutable, content-addressed — version == sha256 of the bytes)

In [ ]:
rows = [
    {"instruction": "What is MLOps-Lite?", "response": "A lightweight local MLOps platform."},
    {"instruction": "What GPU rule does it enforce?", "response": "One model in VRAM at a time."},
    {"instruction": "How are datasets versioned?", "response": "By content hash on MinIO."},
]
jsonl = "\n".join(json.dumps(r) for r in rows)
ds = mlops.register_dataset("qa-demo-nb", jsonl, fmt="jsonl")
print(f"registered {ds['name']} @ {ds['version']}  ({ds['size_bytes']} B, sha {ds['sha256'][:12]}...)")
# re-registering identical bytes dedupes to the SAME version (idempotent):
ds2 = mlops.register_dataset("qa-demo-nb", jsonl, fmt="jsonl")
print("same bytes -> same version:", ds["version"] == ds2["version"])

## 4 · LoRA fine-tune — launch and watch it live

Needs the GPU free (no model resident in serving). If you ran inference above, the model is resident
for ~120 s — `launch_run(wait_for_gpu=True)` polls until it releases. A 5-step run takes ~1–2 min.

In [ ]:
run = mlops.launch_run(dataset_name=ds["name"], dataset_version=ds["version"],
                       output_name="qwen0_5b-qa-nb", steps=5, lora_r=8, seed=0)
run_id = run["run_id"]
print("launched", run_id, "->", run["status"])

last = None
for ev in mlops.stream_run(run_id):
    if ev.get("status") != last:
        last = ev.get("status")
        print("  status:", last)
    if last in ("completed", "failed"):
        break

final = mlops.run(run_id)
print("\nfinal status:", final.get("status"))
if final.get("model"):
    print("registered model:", final["model"]["name"], "v" + str(final["model"]["version"]))
if final.get("metrics"):
    print("metrics:", final["metrics"])
if final.get("error"):
    print("error:", final["error"])

### Promote the new version in the registry

`promote` points the `serving` alias at this version (lineage/traceability). Note: live-serving a
*different* GGUF would also need the supervisor to load it — the configured serving model is the 7B;
this step manages the registry pointer, which is what inference reports as `registry_version`.

In [ ]:
if final.get("model"):
    name, ver = final["model"]["name"], final["model"]["version"]
    mlops.promote(name, ver)
    print(f"promoted {name} v{ver}")
    print("serving pointer:", mlops.model(name)["serving"])

## 5 · Drift monitoring (PSI per feature; optionally trigger a retrain)

In [ ]:
dsets = {d["name"]: d for d in mlops.datasets()}
iris = dsets.get("iris-demo")
if iris and len(iris["versions"]) >= 2:
    a, b = iris["versions"][0]["version"], iris["versions"][1]["version"]
    rep = mlops.drift_check("iris-demo", a, "iris-demo", b, threshold=0.25)["report"]
    print("dataset_drift:", rep["dataset_drift"], "| max_psi:", rep["max_psi"])
    print("per-feature PSI:", rep["features"])
    # to auto-retrain on breach, pass retrain={dataset_name, dataset_version, output_name, steps, lora_r}
else:
    print("Need two versions of a numeric CSV dataset. Register one with mlops.register_dataset(name, csv_bytes, 'csv').")

---
### Notes
- **Where to run:** Windows (VS Code / Jupyter) or WSL — `localhost:8080` reaches the gateway from
  both. Colab/remote can't reach a localhost platform.
- **No heavy deps:** the notebook only needs `requests`; torch/transformers/llama.cpp all run
  server-side in the platform's daemons.
- **Auth:** the key is read from `../.env` (`GATEWAY_API_KEYS`) — keep it out of the notebook and out
  of git (the repo `.gitignore` already excludes `.env`).
- **Mutual exclusion:** serving and training can't both hold the GPU — that's Principle II, surfaced
  as a `409` the client waits through.